# RegLens · Extend to a new cell type by training a ChromBPNet model

**Framing (honest):** we do **not** claim a from-scratch model beats the pretrained
ENCODE/Corces models — it won't in a few days. This notebook demonstrates a different,
real capability: **RegLens extends to any cell type, including ones with *no* public
ChromBPNet model**, by running the standard `chrombpnet pipeline` on that cell type's
ATAC-seq. The trained model then drops into RegLens with zero code changes
(`load_backend(<model.h5>)`).

We demonstrate on **K562 ATAC (ENCODE ENCSR868FGK)** because a *published* K562 model
exists there — so you can validate that this pipeline reproduces sane behavior against
a known-good model. **To extend to a cell type with no public model, just swap the
ENCODE accessions** in the config cell for that cell type's ATAC BAM + peaks.

RegLens's demo and validation run on **pretrained** models regardless — this is the
parallel, non-critical-path extensibility track (spec golden rule #1).

### Requirements (read before running)
- **GPU runtime** (Runtime → Change runtime type → GPU). Colab **Pro** recommended.
- **Time:** several hours **per fold** (~12 h overnight budget).
- **Disk:** tens of GB (ATAC BAM + hg38 + intermediates).
- All inputs are open (ENCODE / UCSC / Kundaje bias model). We do **not** reimplement
  ChromBPNet — we invoke the `chrombpnet` package.

> The `chrombpnet` CLI evolves — `--help` cells are included so you can confirm the
> exact flags in the installed version before the long run.

In [ ]:
# --- GPU check + install ------------------------------------------------------
!nvidia-smi -L || echo 'NO GPU — switch Runtime to GPU before continuing.'
# chrombpnet pulls TensorFlow-GPU and its bioinformatics deps.
!pip -q install chrombpnet
!pip -q install samtools 2>/dev/null; apt-get -qq install -y samtools bedtools >/dev/null 2>&1 || true
import shutil; print('chrombpnet:', shutil.which('chrombpnet'))

In [ ]:
# --- Config: ENCODE accessions + paths ---------------------------------------
# EXAMPLE defaults for K562 ATAC (ENCSR868FGK). VERIFY on the experiment page that the
# BAM and peaks come from the SAME default analysis:
#   https://www.encodeproject.org/experiments/ENCSR868FGK/
import os
CELLTYPE   = 'K562'
BAM_ACC    = 'ENCFF128WZG'   # filtered 'alignments' BAM
PEAKS_ACC  = 'ENCFF057UYP'   # 'replicated peaks' (bed narrowPeak, 10-col)
OUT        = '/content/chrombpnet_run'
DATA       = '/content/data'
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)

def encode_url(acc, ext):
    return f'https://www.encodeproject.org/files/{acc}/@@download/{acc}.{ext}'

In [ ]:
# --- Download inputs ----------------------------------------------------------
%cd $DATA
# hg38 genome + chrom sizes (UCSC).
![ -f hg38.fa ] || (wget -q https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz && gunzip hg38.fa.gz)
![ -f hg38.chrom.sizes ] || wget -q https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes
!samtools faidx hg38.fa
# ENCODE hg38 blacklist.
![ -f blacklist.bed ] || (wget -q -O bl.bed.gz https://github.com/Boyle-Lab/Blacklist/raw/master/lists/hg38-blacklist.v2.bed.gz && gunzip -c bl.bed.gz > blacklist.bed)
# Pretrained Tn5 bias model (K562 ATAC bias) — reused so we skip bias training.
![ -f bias.h5 ] || wget -q -O bias.h5 http://mitra.stanford.edu/kundaje/anusri/chrombpnet_downloads/bias.h5
# ATAC reads + peaks from ENCODE (uses the accessions in the config cell).
import subprocess
if not os.path.exists('atac.bam'):
    subprocess.run(['wget','-q','-O','atac.bam', encode_url(BAM_ACC,'bam')], check=True)
if not os.path.exists('peaks.bed.gz'):
    subprocess.run(['wget','-q','-O','peaks.bed.gz', encode_url(PEAKS_ACC,'bed.gz')], check=True)
!gunzip -kf peaks.bed.gz && head -1 peaks.bed && wc -l peaks.bed
!samtools quickcheck atac.bam && echo 'BAM OK' && ls -lh $DATA

In [ ]:
# --- Fold split (train/valid/test chromosomes) -------------------------------
# ChromBPNet fold JSON: hold out chr1 (test) + chr8,chr10 (valid); train on the rest.
import json
main = [f'chr{i}' for i in list(range(1,23))] + ['chrX']
fold = {'test': ['chr1'], 'valid': ['chr8','chr10'],
        'train': [c for c in main if c not in ('chr1','chr8','chr10')]}
with open(f'{DATA}/fold_0.json','w') as f: json.dump(fold, f)
print(fold)

In [ ]:
# --- Confirm CLI flags (versions drift) + prep GC-matched nonpeaks -----------
!chrombpnet prep nonpeaks --help | head -40
# Generate background (nonpeak) regions GC-matched to the peaks for this fold.
!chrombpnet prep nonpeaks \
  -g $DATA/hg38.fa -p $DATA/peaks.bed -c $DATA/hg38.chrom.sizes \
  -fl $DATA/fold_0.json -br $DATA/blacklist.bed -o $DATA/output

In [ ]:
# --- Train the ChromBPNet model (one fold) -----------------------------------
# Long-running (hours). Confirm flags first if the version differs.
!chrombpnet pipeline --help | head -60
!chrombpnet pipeline \
  -ibam $DATA/atac.bam -d ATAC \
  -g $DATA/hg38.fa -c $DATA/hg38.chrom.sizes \
  -p $DATA/peaks.bed -n $DATA/output_negatives.bed \
  -fl $DATA/fold_0.json -b $DATA/bias.h5 \
  -o $OUT

In [ ]:
# --- Locate the trained model + plug into RegLens ----------------------------
import glob
hits = glob.glob(f'{OUT}/**/chrombpnet_nobias.h5', recursive=True)
print('trained nobias model(s):', hits)
# Then, with the RegLens repo installed on this Colab:
#   from reglens.tools.chrombpnet_score import load_backend, ChromBPNetScorer
#   backend = load_backend(hits[0])          # our trained model
#   # or point at the run dir to average folds once you train more than one.
# Download the .h5 to reuse locally:
from google.colab import files  # noqa
if hits: files.download(hits[0])

## Notes & troubleshooting
- **Verify the BAM/peaks pair** are from the same ENCODE *default analysis* (the
  experiment page groups matched files); a mismatched pair trains on inconsistent data.
- **`chrombpnet prep nonpeaks` / `pipeline` flags can change between versions** — the
  `--help` cells above show the installed version's exact arguments; adjust if needed
  (e.g. the negatives output filename, or `-itag`/`-ifrag` instead of `-ibam`).
- **Bias model:** we reuse the pretrained K562 ATAC `bias.h5`. To train your own bias
  model instead, run `chrombpnet bias pipeline ...` first and pass its `bias.h5`.
- **More folds = better averaging:** train fold_1..fold_4 (swap the held-out chroms) and
  point `reglens ... --model <run_dir>` at all of them for fold+RC averaging.
- **Validation (Saturday):** feed this model's variant Δ-scores into the AUROC harness
  (MPRA/CAGI regulatory vs benign) vs a conservation/CADD baseline.